<a href="https://colab.research.google.com/github/IanCastillo0621/tsp-bio-inspired-algorithms/blob/main/Algoritmo_Exacto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Data from GitHub

In [1]:
!git clone https://github.com/IanCastillo0621/tsp-bio-inspired-algorithms.git

Cloning into 'tsp-bio-inspired-algorithms'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 22 (delta 2), reused 11 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 8.96 MiB | 16.62 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [5]:
!pip install gamspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.4/234.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.5/30.5 MB 12.3 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
from collections import Counter

In [9]:
from gamspy import Container, Set, Parameter, Variable, Equation, Model, Sum, Alias, Options, Ord, Card
import gamspy as gp



# Generate Distance Matrix (matriz_distancias.csv)

The indices were chosen to serve as unique identifiers for each address and are referred to throughout the code as the address unique identifier index.

In [ ]:
df = pd.read_csv('/content/tsp-bio-inspired-algorithms/Data_for_algorithms/Addresses_Jalisco_Mexico.csv')
df

,colonia,Calle,Num Cas,CP,Estado,Full Address,Latitude,Longitude
0,UNIDAD HABITACIONAL FOVISSSTE ESTADIO,LISBOA,28,44307,Guadalajara,UNIDAD HABITACIONAL FOVISSSTE ESTADIO LISBOA 2...,20.710556,-103.324800
1,INDEPENDENCIA,ROGELIO BACON,2280,44290,Guadalajara,INDEPENDENCIA ROGELIO BACON 2280 44290 Guadala...,20.704919,-103.333184
2,BOSQUES DE LA CANTERA,PRIV OPALO,17,44306,Guadalajara,BOSQUES DE LA CANTERA PRIV OPALO 17 44306 Guad...,20.715553,-103.315642
3,JARDINES DE LOS POETAS,JOSE FERNANDEZ,3365,44820,Guadalajara,JARDINES DE LOS POETAS JOSE FERNANDEZ 3365 448...,20.648905,-103.293064
4,LOMAS DEL GALLO,FERNANDO SOLIS,1338,44760,Guadalajara,LOMAS DEL GALLO FERNANDO SOLIS 1338 44760 Guad...,20.679106,-103.270528
...,...,...,...,...,...,...,...,...
424,MESA COLORADA PTE,DELLI,1825,45189,Jalisco,MESA COLORADA PTE DELLI 1825 45189 Jalisco,20.770500,-103.347956
425,VILLA FONTANA DIAMANTE,AV LAS TORRES,1896,45200,Jalisco,VILLA FONTANA DIAMANTE AV LAS TORRES 1896 4520...,20.776952,-103.471263
426,REAL DE TESISTAN,PASEO DE LA ARB,176,45200,Jalisco,REAL DE TESISTAN PASEO DE LA ARB 176 45200 Jal...,20.782630,-103.472207
427,EL CENTINELA 2,HIDALGO,310,45187,Jalisco,EL CENTINELA 2 HIDALGO 310 45187 Jalisco,20.758818,-103.379668


In [ ]:
# Create distance matrix (matriz_distancias.csv)
n = len(df)
matriz_dist = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        # Eucledian distance Lat-Lon (Aproximate)
        x1, y1 = df.loc[i, ['Latitude', 'Longitude']]
        x2, y2 = df.loc[j, ['Latitude', 'Longitude']]
        matriz_dist[i, j] = np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

# Convert to a DataFrame using the original dataset indices as both row and column labels.
df_matriz_dist = pd.DataFrame(matriz_dist, index=df.index, columns=df.index)

print(df_matriz_dist)


          0         1         2         3         4         5         6    \
0    0.000000  0.010103  0.010432  0.069341  0.062726  0.027534  0.071092   
1    0.010103  0.000000  0.020513  0.068901  0.067766  0.019582  0.074029   
2    0.010432  0.020513  0.000000  0.070369  0.057998  0.036466  0.068528   
3    0.069341  0.068901  0.070369  0.000000  0.037683  0.056842  0.024111   
4    0.062726  0.067766  0.057998  0.037683  0.000000  0.066255  0.017227   
..        ...       ...       ...       ...       ...       ...       ...   
424  0.064261  0.067224  0.063745  0.133412  0.119783  0.085661  0.131935   
425  0.160810  0.155738  0.167295  0.219433  0.223312  0.162831  0.229297   
426  0.164084  0.159268  0.170329  0.223551  0.226698  0.166850  0.233002   
427  0.073074  0.071175  0.077274  0.139934  0.135151  0.085004  0.144149   
428  0.215578  0.207902  0.224265  0.259932  0.273955  0.208107  0.275530   

          7         8         9    ...       419       420       421  \
0  

# Defining Functions (Counter/Reconstruct_Path)

In [ ]:
# Function that counts the number of appearances of elements within a list.
def count(list):
  # Count instances
  conteo = Counter(list)

  # Print unique values and number of appearances.
  for valor, cantidad in conteo.items():
      if cantidad > 1:
          print(f"Valor {valor} se repite {cantidad} veces")

  # Print number of unique values
  print(f"Cantidad de valores únicos: {len(conteo)}")
  return

In [ ]:
# Reconstructs a TSP tour by iteratively following (i → j) edges in a DataFrame starting from the first node.

def reconstruct_tsp_path_safe(camino_df):
  camino_df.reset_index(inplace=True, drop = True)

  count = 0

  path = [camino_df["i"][0]]

  while count != camino_df.shape[0]:
    path.append(camino_df[camino_df["i"] == path[-1]]["j"].values[0])
    count += 1

  return path

# Define Function (Random submatrix generator).

In [ ]:

def generar_submatrices_distancias(df_dist, n_subconjuntos=10, tamaño_sub=40, seed=None):
    if seed is not None:
        np.random.seed(seed)

    # Choose a defined amount of random address unique identifiers indexes.
    ubicaciones = df_dist.index.tolist()
    # Store random address unique identifiers indexes that where chosen.
    subconjuntos = []
    # Store Distance matrices.
    matrices = []

    for i in range(n_subconjuntos):
        seleccion = np.random.choice(ubicaciones, tamaño_sub, replace=False)
        submat = df_dist.loc[seleccion, seleccion].copy()

        # Due to precision, addresses close to eachother showed a distance of 0. We changed those values to 0.1 km.
        submat = submat.mask((submat == 0) & (np.eye(tamaño_sub) == 0), 0.1)

        # Change diagonal to 9999 km to reperesent actions the tourist can't do, such as stay in the same address.
        np.fill_diagonal(submat.values, 9999)

        subconjuntos.append(seleccion)
        matrices.append(submat)

        print(f"✅ Subconjunto {i+1}: {len(seleccion)} ubicaciones seleccionadas")

    return matrices, subconjuntos


# Solving instances of 40 addresses

## Generate distance sub matrices (matrices_40.xlsx).

In [ ]:
# Generate 10 instances of sub matrices consisting of 40 randomly chosen addresses. (matrices_40.xlsx)
matrices_40, subconjuntos = generar_submatrices_distancias(df_matriz_dist)


✅ Subconjunto 1: 40 ubicaciones seleccionadas
✅ Subconjunto 2: 40 ubicaciones seleccionadas
✅ Subconjunto 3: 40 ubicaciones seleccionadas
✅ Subconjunto 4: 40 ubicaciones seleccionadas
✅ Subconjunto 5: 40 ubicaciones seleccionadas
✅ Subconjunto 6: 40 ubicaciones seleccionadas
✅ Subconjunto 7: 40 ubicaciones seleccionadas
✅ Subconjunto 8: 40 ubicaciones seleccionadas
✅ Subconjunto 9: 40 ubicaciones seleccionadas
✅ Subconjunto 10: 40 ubicaciones seleccionadas


In [ ]:
for i in matrices_40:
  display(i)

,143,186,4,37,197,267,24,217,215,259,...,387,178,32,213,159,273,375,304,211,348
143,9999.000000,0.150940,0.044168,0.054234,0.151789,0.392502,0.016902,0.119113,0.100696,0.134617,...,0.152908,0.129240,0.057463,0.106890,0.123275,0.218469,0.107404,0.102064,0.151445,0.199640
186,0.150940,9999.000000,0.191741,0.107635,0.023971,0.292533,0.143554,0.035640,0.057062,0.016510,...,0.008116,0.081714,0.186236,0.157396,0.043810,0.197776,0.044580,0.067055,0.008136,0.234944
4,0.044168,0.191741,9999.000000,0.098318,0.194549,0.435474,0.058737,0.161575,0.144082,0.175247,...,0.194434,0.173146,0.069878,0.140571,0.166819,0.255511,0.147360,0.146149,0.191390,0.226529
37,0.054234,0.107635,0.098318,9999.000000,0.103108,0.339102,0.040665,0.072621,0.051718,0.092646,...,0.107633,0.075094,0.078627,0.076479,0.072683,0.174416,0.069905,0.049223,0.110254,0.172204
197,0.151789,0.023971,0.194549,0.103108,9999.000000,0.272909,0.142031,0.033089,0.051653,0.030414,...,0.016271,0.061750,0.180251,0.142128,0.031125,0.174324,0.053449,0.056361,0.032107,0.214520
267,0.392502,0.292533,0.435474,0.339102,0.272909,9999.000000,0.376831,0.299366,0.306242,0.303264,...,0.284854,0.264638,0.386779,0.308887,0.285969,0.206681,0.323856,0.297338,0.299375,0.282542
24,0.016902,0.143554,0.058737,0.040665,0.142031,0.376831,9999.000000,0.110111,0.090404,0.127603,...,0.144714,0.114655,0.050736,0.090414,0.112396,0.201567,0.101560,0.089744,0.144902,0.184046
217,0.119113,0.035640,0.161575,0.072621,0.033089,0.299366,0.110111,9999.000000,0.021434,0.023265,...,0.035013,0.056534,0.151022,0.123749,0.016283,0.178474,0.024941,0.032567,0.039829,0.206348
215,0.100696,0.057062,0.144082,0.051718,0.051653,0.306242,0.090404,0.021434,9999.000000,0.043704,...,0.056211,0.049095,0.129801,0.104562,0.022822,0.170263,0.031311,0.015473,0.060906,0.190921
259,0.134617,0.016510,0.175247,0.092646,0.030414,0.303264,0.127603,0.023265,0.043704,9999.000000,...,0.020813,0.076787,0.171247,0.147013,0.036131,0.196860,0.028091,0.055831,0.017608,0.228630


,17,369,207,335,151,146,11,81,127,154,...,112,364,217,188,36,263,332,99,338,119
17,9999.000000,0.136014,0.100650,0.115875,0.159631,0.059361,0.013499,0.036783,0.046068,0.136273,...,0.038937,0.028810,0.103782,0.054089,0.066391,0.376468,0.210130,0.018827,0.119811,0.067293
369,0.136014,9999.000000,0.135661,0.046196,0.024452,0.191462,0.128980,0.172796,0.181514,0.005100,...,0.102191,0.112831,0.032558,0.129800,0.078211,0.276606,0.325109,0.138341,0.031528,0.071329
207,0.100650,0.135661,9999.000000,0.090864,0.149466,0.149617,0.108813,0.119697,0.131209,0.132326,...,0.110055,0.109721,0.117696,0.046818,0.120265,0.297034,0.200900,0.083983,0.142802,0.110443
335,0.115875,0.046196,0.090864,9999.000000,0.058789,0.175025,0.113182,0.151223,0.161569,0.042225,...,0.092174,0.100597,0.042254,0.093681,0.078520,0.267082,0.285961,0.112193,0.063541,0.067244
151,0.159631,0.024452,0.149466,0.058789,9999.000000,0.215660,0.153032,0.196399,0.205354,0.023372,...,0.126499,0.137068,0.056828,0.149448,0.102652,0.257045,0.343812,0.160970,0.053151,0.095697
146,0.059361,0.191462,0.149617,0.175025,0.215660,9999.000000,0.062870,0.029974,0.018787,0.192358,...,0.089385,0.078637,0.158905,0.104116,0.115522,0.435025,0.193895,0.068626,0.170918,0.120318
11,0.013499,0.128980,0.108813,0.113182,0.153032,0.062870,9999.000000,0.045598,0.052908,0.129685,...,0.028201,0.017288,0.096451,0.063302,0.055797,0.377017,0.223197,0.031206,0.110495,0.058468
81,0.036783,0.172796,0.119697,0.151223,0.196399,0.029974,0.045598,9999.000000,0.011635,0.173047,...,0.073782,0.062884,0.140531,0.074563,0.101394,0.407310,0.184349,0.040577,0.155892,0.103545
127,0.046068,0.181514,0.131209,0.161569,0.205354,0.018787,0.052908,0.011635,9999.000000,0.181983,...,0.080929,0.069939,0.149075,0.086196,0.108239,0.418751,0.185053,0.051919,0.163308,0.111372
154,0.136273,0.005100,0.132326,0.042225,0.023372,0.192358,0.129685,0.173047,0.181983,9999.000000,...,0.103305,0.113807,0.033726,0.128019,0.080104,0.272526,0.323123,0.137904,0.035826,0.072616


,224,131,70,179,105,420,61,273,212,13,...,240,425,15,252,389,71,397,182,63,326
224,9999.000000,0.059430,0.089317,0.063031,0.088770,0.054469,0.133145,0.121494,0.118209,0.113390,...,0.053797,0.087047,0.105441,0.025107,0.085732,0.116010,0.053355,0.049684,0.123312,0.127574
131,0.059430,9999.000000,0.060064,0.086553,0.072001,0.053135,0.089295,0.177370,0.175258,0.066457,...,0.110866,0.141104,0.076019,0.081819,0.028276,0.065355,0.052484,0.088888,0.064243,0.100959
70,0.089317,0.060064,9999.000000,0.066452,0.018507,0.038516,0.047043,0.208371,0.203792,0.035079,...,0.141382,0.175667,0.016927,0.099272,0.050657,0.043162,0.039331,0.134847,0.089482,0.158955
179,0.063031,0.086553,0.066452,9999.000000,0.052990,0.035779,0.112051,0.158264,0.151813,0.101212,...,0.097876,0.131664,0.075393,0.054158,0.098675,0.108438,0.035961,0.111243,0.142775,0.180738
105,0.088770,0.072001,0.018507,0.052990,9999.000000,0.034354,0.059384,0.203468,0.198168,0.052152,...,0.137900,0.172554,0.022414,0.094014,0.067523,0.060856,0.035451,0.136988,0.107735,0.172675
420,0.054469,0.053135,0.038516,0.035779,0.034354,9999.000000,0.085403,0.170449,0.165604,0.070179,...,0.104090,0.138663,0.052813,0.061017,0.062896,0.076086,0.001142,0.103165,0.106996,0.151943
61,0.133145,0.089295,0.047043,0.112051,0.059384,0.085403,9999.000000,0.253912,0.249744,0.023091,...,0.186423,0.220180,0.037206,0.145594,0.065695,0.027947,0.086157,0.174862,0.082881,0.173762
273,0.121494,0.177370,0.208371,0.158264,0.203468,0.170449,0.253912,9999.000000,0.009629,0.234799,...,0.067705,0.036963,0.223261,0.109531,0.205293,0.237483,0.169503,0.094832,0.237119,0.197776
212,0.118209,0.175258,0.203792,0.151813,0.198168,0.165604,0.249744,0.009629,9999.000000,0.231114,...,0.064602,0.037497,0.218353,0.104587,0.202928,0.234149,0.164691,0.095391,0.236040,0.200710
13,0.113390,0.066457,0.035079,0.101212,0.052152,0.070179,0.023091,0.234799,0.231114,9999.000000,...,0.167105,0.200293,0.034403,0.128045,0.042707,0.009600,0.070726,0.153120,0.065713,0.152019


,367,285,24,59,134,421,344,335,51,49,...,96,171,119,33,337,313,140,420,248,40
367,9999.000000,0.215571,0.174785,0.161293,0.207192,0.125832,0.102663,0.132113,0.215643,0.181380,...,0.218625,0.112543,0.176378,0.155002,0.173705,0.119334,0.219842,0.141323,0.188570,0.205492
285,0.215571,9999.000000,0.133831,0.141063,0.110287,0.157215,0.115868,0.084818,0.122793,0.134756,...,0.128095,0.148300,0.061470,0.182046,0.053275,0.101203,0.141983,0.149580,0.028051,0.164551
24,0.174785,0.133831,9999.000000,0.015399,0.042263,0.052955,0.131335,0.120371,0.044555,0.006612,...,0.045942,0.062383,0.074184,0.054318,0.142031,0.087549,0.045068,0.036526,0.126414,0.039665
59,0.161293,0.141063,0.015399,9999.000000,0.057189,0.037808,0.125181,0.117762,0.059951,0.021307,...,0.061227,0.048788,0.079953,0.042149,0.143695,0.082496,0.059279,0.021295,0.130792,0.047035
134,0.207192,0.110287,0.042263,0.057189,9999.000000,0.092926,0.144820,0.125307,0.013718,0.038194,...,0.019184,0.098241,0.062093,0.095671,0.133801,0.101984,0.031965,0.077204,0.111594,0.056643
421,0.125832,0.157215,0.052955,0.037808,0.092926,9999.000000,0.109513,0.111766,0.097307,0.059097,...,0.098896,0.018351,0.096851,0.037792,0.146648,0.073861,0.096899,0.016522,0.140805,0.079662
344,0.102663,0.115868,0.131335,0.125181,0.144820,0.109513,9999.000000,0.031086,0.157243,0.137014,...,0.161968,0.091531,0.092526,0.147125,0.071045,0.044072,0.169454,0.115489,0.088007,0.170741
335,0.132113,0.084818,0.120371,0.117762,0.125307,0.111766,0.031086,9999.000000,0.138550,0.125147,...,0.143675,0.095715,0.067244,0.147375,0.043354,0.037912,0.153069,0.113298,0.057050,0.159883
51,0.215643,0.122793,0.044555,0.059951,0.013718,0.097307,0.157243,0.138550,9999.000000,0.039000,...,0.005497,0.104820,0.075809,0.094645,0.147502,0.113950,0.019297,0.080999,0.125080,0.048164
49,0.181380,0.134756,0.006612,0.021307,0.038194,0.059097,0.137014,0.125147,0.039000,9999.000000,...,0.039977,0.068947,0.076185,0.057498,0.145317,0.093090,0.038493,0.042597,0.128674,0.034778


,114,375,130,391,263,257,347,184,335,318,...,207,161,378,254,302,417,201,422,413,239
114,9999.000000,0.122615,0.080407,0.160893,0.425616,0.086697,0.151023,0.172966,0.160955,0.159121,...,0.151023,0.094454,0.120765,0.233450,0.117079,0.176665,0.070467,0.097572,0.253323,0.145774
375,0.122615,9999.000000,0.047150,0.043518,0.325105,0.113425,0.130285,0.093161,0.065968,0.078037,...,0.130285,0.128431,0.052878,0.184522,0.092768,0.130215,0.125008,0.136009,0.200231,0.130003
130,0.080407,0.047150,9999.000000,0.080819,0.349892,0.071836,0.108040,0.100471,0.082897,0.085099,...,0.108040,0.087047,0.046059,0.178928,0.066628,0.120389,0.079499,0.094384,0.197235,0.105563
391,0.160893,0.043518,0.080819,9999.000000,0.282052,0.131075,0.125328,0.066956,0.038042,0.055965,...,0.125328,0.144547,0.056880,0.160195,0.097524,0.113685,0.148159,0.151924,0.173153,0.127216
263,0.425616,0.325105,0.349892,0.282052,9999.000000,0.356059,0.297034,0.252728,0.267082,0.266561,...,0.297034,0.360117,0.305444,0.226155,0.314413,0.262670,0.379519,0.363869,0.210059,0.303492
257,0.086697,0.113425,0.071836,0.131075,0.356059,9999.000000,0.065909,0.109134,0.110987,0.100231,...,0.065909,0.015235,0.074555,0.149358,0.043075,0.095911,0.023460,0.022704,0.169445,0.060138
347,0.151023,0.130285,0.108040,0.125328,0.297034,0.065909,9999.000000,0.071961,0.090864,0.073134,...,0.100000,0.065102,0.077475,0.083467,0.041416,0.034761,0.087904,0.067606,0.103564,0.006470
184,0.172966,0.093161,0.100471,0.066956,0.252728,0.109134,0.071961,9999.000000,0.029316,0.015809,...,0.071961,0.117488,0.054449,0.093590,0.066095,0.049525,0.131844,0.123308,0.107770,0.076293
335,0.160955,0.065968,0.082897,0.038042,0.267082,0.110987,0.090864,0.029316,9999.000000,0.018398,...,0.090864,0.122278,0.040871,0.122906,0.070954,0.075888,0.131602,0.129072,0.136821,0.093704
318,0.159121,0.078037,0.085099,0.055965,0.266561,0.100231,0.073134,0.015809,0.018398,9999.000000,...,0.073134,0.110100,0.039282,0.107172,0.058075,0.057721,0.122149,0.116465,0.122276,0.076354


,410,260,163,231,28,377,99,278,391,167,...,350,132,384,359,133,358,386,362,275,131
410,9999.000000,0.092944,0.019514,0.166947,0.050624,0.021020,0.089749,0.134041,0.036705,0.106025,...,0.090180,0.104975,0.028319,0.161167,0.050764,0.201255,0.102871,0.142777,0.120706,0.049025
260,0.092944,9999.000000,0.074051,0.087203,0.107520,0.110667,0.093203,0.085378,0.111475,0.082014,...,0.069583,0.116223,0.064935,0.078365,0.135278,0.108312,0.020440,0.062678,0.041017,0.084936
163,0.019514,0.074051,9999.000000,0.147672,0.058803,0.036925,0.086505,0.123014,0.043920,0.098072,...,0.081388,0.104927,0.009122,0.141746,0.068555,0.182287,0.085312,0.123391,0.101235,0.048872
231,0.166947,0.087203,0.147672,9999.000000,0.192827,0.179208,0.179160,0.154303,0.170697,0.163401,...,0.154091,0.201758,0.139674,0.010870,0.215350,0.076077,0.096279,0.024983,0.047840,0.171104
28,0.050624,0.107520,0.058803,0.192827,9999.000000,0.064687,0.053085,0.111199,0.085598,0.078034,...,0.067330,0.060258,0.061862,0.184939,0.039997,0.209385,0.107900,0.167844,0.145004,0.023336
377,0.021020,0.110667,0.036925,0.179208,0.064687,9999.000000,0.109102,0.155031,0.022308,0.126643,...,0.111001,0.122646,0.045993,0.174360,0.050457,0.218641,0.122196,0.155829,0.134607,0.068136
99,0.089749,0.093203,0.086505,0.179160,0.053085,0.109102,9999.000000,0.061130,0.126218,0.027690,...,0.026113,0.023150,0.083584,0.169501,0.092922,0.176685,0.083520,0.155333,0.134210,0.040967
278,0.134041,0.085378,0.123014,0.154303,0.111199,0.155031,0.061130,9999.000000,0.166876,0.033641,...,0.045094,0.072389,0.116277,0.143524,0.150951,0.128998,0.066009,0.134776,0.119321,0.092503
391,0.036705,0.111475,0.043920,0.170697,0.085598,0.022308,0.126218,0.166876,9999.000000,0.140924,...,0.124533,0.141652,0.051456,0.167057,0.072264,0.217113,0.125974,0.148635,0.128933,0.085682
167,0.106025,0.082014,0.098072,0.163401,0.078034,0.126643,0.027690,0.033641,0.140924,9999.000000,...,0.016933,0.042971,0.092833,0.153155,0.117969,0.152509,0.067470,0.140922,0.121659,0.060868


,309,91,112,412,246,413,283,61,291,28,...,116,167,81,384,186,30,372,89,96,375
309,9999.000000,0.132404,0.156457,0.086896,0.051776,0.053057,0.104605,0.202267,0.161215,0.153447,...,0.168404,0.118639,0.170681,0.106834,0.172763,0.193598,0.033157,0.150427,0.193467,0.164679
91,0.132404,9999.000000,0.093614,0.049841,0.083464,0.185432,0.100812,0.085150,0.196766,0.095992,...,0.124037,0.022877,0.047975,0.115700,0.181539,0.119862,0.160859,0.116180,0.091695,0.143883
112,0.156457,0.093614,9999.000000,0.086841,0.110011,0.202465,0.058329,0.075762,0.126499,0.005548,...,0.031239,0.076550,0.073782,0.066923,0.100183,0.037246,0.170150,0.030934,0.051802,0.056499
412,0.086896,0.049841,0.086841,9999.000000,0.035446,0.139579,0.064178,0.116268,0.158723,0.086251,...,0.110554,0.032053,0.083786,0.077536,0.150758,0.122340,0.112721,0.096599,0.111034,0.121340
246,0.051776,0.083464,0.110011,0.035446,9999.000000,0.104186,0.067869,0.150539,0.150236,0.107900,...,0.127398,0.067470,0.119080,0.076354,0.150644,0.147140,0.077565,0.110667,0.142699,0.130657
413,0.053057,0.185432,0.202465,0.139579,0.104186,9999.000000,0.146334,0.253912,0.177177,0.198779,...,0.209518,0.171486,0.223261,0.144317,0.197776,0.238868,0.033027,0.191262,0.243349,0.200231
283,0.104605,0.100812,0.058329,0.064178,0.067869,0.146334,9999.000000,0.125619,0.096310,0.053844,...,0.063932,0.077957,0.108521,0.014941,0.086713,0.093132,0.113513,0.045843,0.106677,0.062847
61,0.202267,0.085150,0.075762,0.116268,0.150539,0.253912,0.125619,9999.000000,0.202056,0.081273,...,0.100573,0.086901,0.037206,0.137813,0.173762,0.070772,0.224414,0.106366,0.027567,0.129188
291,0.161215,0.196766,0.126499,0.158723,0.150236,0.177177,0.096310,0.202056,9999.000000,0.121083,...,0.103997,0.174011,0.196399,0.081761,0.036513,0.141769,0.148600,0.095697,0.176527,0.075732
28,0.153447,0.095992,0.005548,0.086251,0.107900,0.198779,0.053844,0.081273,0.121083,9999.000000,...,0.028080,0.078034,0.078356,0.061862,0.095190,0.040225,0.166321,0.025675,0.057326,0.051873


,29,126,100,245,239,131,10,61,304,164,...,201,254,307,160,227,425,415,48,62,370
29,9999.000000,0.031069,0.027920,0.061970,0.104093,0.057067,0.016251,0.041503,0.102030,0.102030,...,0.030141,0.192557,0.046761,0.142416,0.184867,0.179175,0.128772,0.036425,0.015794,0.074586
126,0.031069,9999.000000,0.053346,0.046074,0.081065,0.028351,0.041469,0.069217,0.072152,0.072152,...,0.036203,0.166618,0.038840,0.121420,0.161181,0.151723,0.101088,0.034285,0.033538,0.069797
100,0.027920,0.053346,9999.000000,0.089824,0.131305,0.073008,0.037109,0.016314,0.117023,0.117023,...,0.057006,0.219023,0.074616,0.143542,0.212115,0.204799,0.154181,0.064112,0.041940,0.101457
245,0.061970,0.046074,0.089824,9999.000000,0.044293,0.060804,0.059352,0.103058,0.088121,0.088121,...,0.037556,0.133695,0.017736,0.153845,0.123973,0.122969,0.075119,0.028256,0.050660,0.030130
239,0.104093,0.081065,0.131305,0.044293,9999.000000,0.081819,0.103377,0.145594,0.086097,0.086097,...,0.081831,0.089426,0.062012,0.159155,0.080814,0.079175,0.035124,0.072494,0.094414,0.058100
131,0.057067,0.028351,0.073008,0.060804,0.081819,9999.000000,0.069429,0.089295,0.045052,0.045052,...,0.063571,0.158497,0.060868,0.095337,0.156837,0.141104,0.091829,0.059739,0.061843,0.089454
10,0.016251,0.041469,0.037109,0.059352,0.103377,0.069429,9999.000000,0.046204,0.113616,0.113616,...,0.022334,0.192757,0.042099,0.157930,0.183298,0.180909,0.131298,0.031212,0.009461,0.065336
61,0.041503,0.069217,0.016314,0.103058,0.145594,0.089295,0.046204,9999.000000,0.133139,0.133139,...,0.068010,0.233950,0.086901,0.156375,0.226355,0.220180,0.169623,0.076077,0.053155,0.111519
304,0.102030,0.072152,0.117023,0.088121,0.086097,0.045052,0.113616,0.133139,9999.000000,0.100000,...,0.104258,0.138768,0.095691,0.073268,0.143339,0.118418,0.077455,0.098271,0.105409,0.118115
164,0.102030,0.072152,0.117023,0.088121,0.086097,0.045052,0.113616,0.133139,0.100000,9999.000000,...,0.104258,0.138768,0.095691,0.073268,0.143339,0.118418,0.077455,0.098271,0.105409,0.118115


,411,298,266,178,145,48,290,190,112,122,...,19,349,347,111,42,295,224,375,389,185
411,9999.000000,0.136100,0.074247,0.114949,0.085732,0.029507,0.162873,0.107827,0.096076,0.099727,...,0.095232,0.158238,0.061069,0.068096,0.073960,0.152893,0.059756,0.139968,0.089695,0.017898
298,0.136100,9999.000000,0.086297,0.106088,0.198676,0.156909,0.152265,0.136863,0.180504,0.228935,...,0.222051,0.024134,0.078612,0.198250,0.171415,0.159308,0.097794,0.182935,0.182660,0.121283
266,0.074247,0.086297,9999.000000,0.046732,0.114204,0.082604,0.101150,0.060009,0.094319,0.148973,...,0.141044,0.110384,0.026895,0.120580,0.085743,0.097921,0.015520,0.106013,0.096406,0.069175
178,0.114949,0.106088,0.046732,9999.000000,0.126385,0.114367,0.055020,0.035543,0.093287,0.168013,...,0.158897,0.127633,0.073134,0.144772,0.096324,0.055378,0.056874,0.078037,0.100217,0.113552
145,0.085732,0.198676,0.114204,0.126385,9999.000000,0.056393,0.148868,0.096575,0.040632,0.044252,...,0.034652,0.222732,0.121689,0.037392,0.030201,0.131288,0.101051,0.096383,0.029934,0.102506
48,0.029507,0.156909,0.082604,0.114367,0.056393,9999.000000,0.155353,0.098271,0.072034,0.073023,...,0.067421,0.180105,0.078546,0.041601,0.048859,0.142441,0.067097,0.122034,0.064005,0.046889
290,0.162873,0.152265,0.101150,0.055020,0.148868,0.155353,9999.000000,0.057157,0.108862,0.193101,...,0.183507,0.170683,0.127886,0.176662,0.122417,0.019686,0.109114,0.063314,0.119048,0.164595
190,0.107827,0.136863,0.060009,0.035543,0.096575,0.098271,0.057157,9999.000000,0.059887,0.140054,...,0.130597,0.159612,0.085479,0.120791,0.067830,0.045080,0.061368,0.046753,0.068203,0.112046
112,0.096076,0.180504,0.094319,0.093287,0.040632,0.072034,0.108862,0.059887,9999.000000,0.084739,...,0.075149,0.204517,0.110055,0.073782,0.023255,0.090877,0.085083,0.056499,0.010990,0.108987
122,0.099727,0.228935,0.148973,0.168013,0.044252,0.073023,0.193101,0.140054,0.084739,9999.000000,...,0.009601,0.252459,0.150327,0.031688,0.072350,0.175525,0.134288,0.139583,0.074168,0.117591


,76,45,420,225,331,401,122,345,127,318,...,34,92,105,309,162,409,65,106,204,177
76,9999.000000,0.119071,0.025569,0.171419,0.111364,0.165159,0.066837,0.156343,0.045482,0.127757,...,0.085363,0.005986,0.012932,0.138734,0.070723,0.156202,0.109276,0.056692,0.141050,0.026967
45,0.119071,9999.000000,0.107550,0.202934,0.069636,0.085881,0.109925,0.213086,0.111978,0.104939,...,0.034203,0.124913,0.114278,0.182739,0.099750,0.059013,0.015315,0.062996,0.124204,0.121965
420,0.025569,0.107550,9999.000000,0.149590,0.088307,0.142998,0.083631,0.138716,0.064345,0.102455,...,0.073431,0.028141,0.034354,0.117935,0.045158,0.137028,0.100504,0.045394,0.115489,0.014612
225,0.171419,0.202934,0.149590,9999.000000,0.134587,0.159947,0.232684,0.042235,0.213934,0.102432,...,0.182450,0.170528,0.182935,0.033861,0.115604,0.181971,0.207383,0.171456,0.086952,0.144497
331,0.111364,0.069636,0.088307,0.134587,9999.000000,0.055095,0.140054,0.149725,0.130711,0.035543,...,0.059237,0.115464,0.114386,0.117950,0.052702,0.057396,0.077100,0.067490,0.055030,0.098303
401,0.165159,0.085881,0.142998,0.159947,0.055095,9999.000000,0.184119,0.185811,0.178709,0.063245,...,0.096764,0.169597,0.166845,0.153656,0.106918,0.030957,0.099831,0.115581,0.073595,0.153370
122,0.066837,0.109925,0.083631,0.232684,0.140054,0.184119,9999.000000,0.221816,0.021757,0.168013,...,0.087496,0.070620,0.053906,0.201502,0.120574,0.164476,0.095116,0.072892,0.185300,0.092043
345,0.156343,0.213086,0.138716,0.042235,0.149725,0.185811,0.221816,9999.000000,0.201340,0.123548,...,0.187288,0.154065,0.168911,0.032310,0.116039,0.203266,0.214571,0.170309,0.112823,0.129948
127,0.045482,0.111978,0.064345,0.213934,0.130711,0.178709,0.021757,0.201340,9999.000000,0.155653,...,0.084389,0.048993,0.032610,0.182124,0.104498,0.162189,0.098300,0.063605,0.171910,0.071392
318,0.127757,0.104939,0.102455,0.102432,0.035543,0.063245,0.168013,0.123548,0.155653,9999.000000,...,0.093263,0.130578,0.133833,0.091243,0.058075,0.080254,0.112623,0.095505,0.019534,0.108339


In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 1.8 MB/s eta 0:00:00


In [ ]:
# Save sub matrices to a excel file for later use.
archivo_salida = "matrices_40.xlsx"

with pd.ExcelWriter(archivo_salida, engine="xlsxwriter") as writer:
    for idx, matriz in enumerate(matrices_40):
        hoja_nombre = f"Instancia_{idx+1}"
        matriz.to_excel(writer, sheet_name=hoja_nombre, index=True)

## Read pre-generated submatrices.

In [8]:
archivo = "/content/tsp-bio-inspired-algorithms/Data_for_algorithms/matrices_40.xlsx"

# Read file.
hojas_dict = pd.read_excel(archivo, sheet_name=None, index_col=0)

# Convert lists into dataframes.
matrices_40 = [hojas_dict[f"Instancia_{i+1}"] for i in range(10)]

# Validation.
print(len(matrices_40))  # print number of instances 10.
print(matrices_40[0].head()) # print first instance.
print(matrices_40[0].shape) # print first instance shape.


10
             332          163          115          388          402  \
332  9999.000000     0.269340    66.004588     0.317838     0.257123   
163     0.269340  9999.000000    65.918566     0.053220     0.029585   
115    66.004588    65.918566  9999.000000    65.880899    65.897945   
388     0.317838     0.053220    65.880899  9999.000000     0.061222   
402     0.257123     0.029585    65.897945     0.061222  9999.000000   

           361        41         383        89         418  ...        26   \
332   0.291817   0.280746   0.261074   0.270650   0.146346  ...   0.214347   
163   0.046070   0.079073   0.153423   0.041487   0.123014  ...   0.094960   
115  65.872791  65.843577  66.071954  65.879866  65.956187  ...  65.874983   
388   0.033375   0.072928   0.195391   0.052543   0.172043  ...   0.122073   
402   0.036477   0.054506   0.174931   0.019133   0.112184  ...   0.067193   

           43         277        334        239        351        225  \
332   0.213167   0.231

In [ ]:
for i in matrices_40:
  display(i)

,332,163,115,388,402,361,41,383,89,418,...,26,43,277,334,239,351,225,92,271,15
332,9999.000000,0.269340,66.004588,0.317838,0.257123,0.291817,0.280746,0.261074,0.270650,0.146346,...,0.214347,0.213167,0.231830,0.199969,0.196066,0.337408,0.252114,0.160885,0.160022,0.184349
163,0.269340,9999.000000,65.918566,0.053220,0.029585,0.046070,0.079073,0.153423,0.041487,0.123014,...,0.094960,0.070429,0.081178,0.085312,0.085725,0.071857,0.131513,0.118168,0.109604,0.124407
115,66.004588,65.918566,9999.000000,65.880899,65.897945,65.872791,65.843577,66.071954,65.879866,65.956187,...,65.874983,65.901805,65.996210,65.982412,65.978857,65.872865,66.049866,65.920244,65.947616,65.880752
388,0.317838,0.053220,65.880899,9999.000000,0.061222,0.033375,0.072928,0.195391,0.052543,0.172043,...,0.122073,0.108022,0.130594,0.138476,0.138944,0.019576,0.174883,0.160614,0.158033,0.156074
402,0.257123,0.029585,65.897945,0.061222,9999.000000,0.036477,0.054506,0.174931,0.019133,0.112184,...,0.067193,0.047341,0.098296,0.091566,0.090129,0.080649,0.152544,0.099455,0.097973,0.098802
361,0.291817,0.046070,65.872791,0.033375,0.036477,9999.000000,0.042078,0.199344,0.021360,0.148020,...,0.089575,0.079105,0.126757,0.125830,0.124994,0.050224,0.177529,0.132140,0.133748,0.123945
41,0.280746,0.079073,65.843577,0.072928,0.054506,0.042078,9999.000000,0.229417,0.037678,0.145388,...,0.067053,0.072554,0.152640,0.142353,0.139939,0.085659,0.207043,0.120530,0.131566,0.100504
383,0.261074,0.153423,66.071954,0.195391,0.174931,0.199344,0.229417,9999.000000,0.192442,0.162446,...,0.213041,0.182209,0.078448,0.104232,0.109535,0.207593,0.022479,0.190369,0.161147,0.223310
89,0.270650,0.041487,65.879866,0.052543,0.019133,0.021360,0.037678,0.192442,9999.000000,0.127498,...,0.069570,0.057766,0.116671,0.110667,0.109142,0.070728,0.170166,0.110791,0.113210,0.103545
418,0.146346,0.123014,65.956187,0.172043,0.112184,0.148020,0.145388,0.162446,0.127498,9999.000000,...,0.087536,0.072987,0.101699,0.066009,0.060614,0.191611,0.144307,0.035958,0.014288,0.075449


,88,33,327,379,418,137,369,392,249,361,...,56,152,210,72,310,24,122,345,346,403
88,9999.000000,0.027717,0.152298,0.065644,0.037905,0.114868,0.152506,0.066516,0.062115,0.114835,...,0.041758,0.162546,0.018279,0.058707,0.139178,0.035719,0.078892,0.142939,0.132615,0.364884
33,0.027717,9999.000000,0.179668,0.089065,0.027868,0.136043,0.180170,0.080884,0.083509,0.141888,...,0.057401,0.175406,0.044301,0.085955,0.166632,0.054318,0.079307,0.149250,0.160213,0.383984
327,0.152298,0.179668,9999.000000,0.119048,0.175820,0.126434,0.015952,0.130697,0.109427,0.066281,...,0.150837,0.149864,0.135528,0.100705,0.013476,0.145680,0.193101,0.174385,0.043548,0.262498
379,0.065644,0.089065,0.119048,9999.000000,0.103262,0.050096,0.112831,0.106132,0.086632,0.061302,...,0.037408,0.190328,0.061896,0.028276,0.105886,0.036276,0.074168,0.185119,0.084897,0.366561
418,0.037905,0.027868,0.175820,0.103262,9999.000000,0.152740,0.178784,0.060614,0.070357,0.148020,...,0.078059,0.151293,0.045094,0.092503,0.163502,0.072945,0.106475,0.122452,0.163011,0.363434
137,0.114868,0.136043,0.126434,0.050096,0.152740,9999.000000,0.114709,0.153580,0.131994,0.060353,...,0.079368,0.230296,0.111976,0.071835,0.115649,0.081729,0.094700,0.230718,0.083527,0.387200
369,0.152506,0.180170,0.015952,0.112831,0.178784,0.114709,9999.000000,0.137839,0.115307,0.055744,...,0.146671,0.164217,0.136642,0.097730,0.018052,0.142171,0.186929,0.186735,0.031228,0.276606
392,0.066516,0.080884,0.130697,0.106132,0.060614,0.153580,0.137839,9999.000000,0.024383,0.124994,...,0.101712,0.096093,0.052980,0.081819,0.120421,0.094000,0.144588,0.079868,0.131371,0.303492
249,0.062115,0.083509,0.109427,0.086632,0.070357,0.131994,0.115307,0.024383,9999.000000,0.100664,...,0.089319,0.106084,0.044723,0.060329,0.098349,0.081464,0.135032,0.098866,0.107238,0.302779
361,0.114835,0.141888,0.066281,0.061302,0.148020,0.060353,0.055744,0.124994,0.100664,9999.000000,...,0.097985,0.183055,0.102954,0.056154,0.055301,0.095163,0.133835,0.192766,0.024977,0.327061


,191,178,375,376,380,47,337,286,144,162,...,117,368,391,264,0,154,366,151,157,150
191,9999.000000,0.035543,0.046753,0.083780,0.095997,0.072003,0.056361,0.123902,0.060786,0.052702,...,0.018004,0.114953,0.045541,0.028991,0.099494,0.056123,0.118065,0.078473,0.118418,0.057157
178,0.035543,9999.000000,0.078037,0.086693,0.069722,0.088661,0.061750,0.091662,0.096304,0.058075,...,0.048459,0.117633,0.055965,0.042039,0.118988,0.060622,0.120884,0.076361,0.085985,0.055020
375,0.046753,0.078037,9999.000000,0.121340,0.142712,0.099109,0.053449,0.169299,0.039140,0.092768,...,0.050367,0.150253,0.043518,0.041882,0.118621,0.054548,0.153062,0.075732,0.163692,0.063314
376,0.083780,0.086693,0.121340,9999.000000,0.073984,0.033658,0.138764,0.111549,0.107466,0.031257,...,0.071892,0.031567,0.128775,0.111848,0.053396,0.138238,0.034779,0.158723,0.108656,0.136579
380,0.095997,0.069722,0.142712,0.073984,9999.000000,0.099990,0.130638,0.037795,0.148982,0.069676,...,0.098250,0.092275,0.125647,0.111469,0.126298,0.129368,0.094901,0.141550,0.034673,0.122184
47,0.072003,0.088661,0.099109,0.033658,0.099990,9999.000000,0.128126,0.137609,0.077910,0.034030,...,0.055491,0.053805,0.116892,0.100855,0.030554,0.127987,0.056147,0.150474,0.133878,0.129003
337,0.056361,0.061750,0.053449,0.138764,0.130638,0.128126,9999.000000,0.146536,0.090886,0.107526,...,0.072810,0.170267,0.011814,0.027372,0.154444,0.001648,0.173441,0.023789,0.140801,0.012591
286,0.123902,0.091662,0.169299,0.111549,0.037795,0.137609,0.146536,9999.000000,0.181168,0.106379,...,0.129821,0.127393,0.144313,0.132884,0.164084,0.145046,0.129748,0.151893,0.005756,0.135961
144,0.060786,0.096304,0.039140,0.107466,0.148982,0.077910,0.090886,0.181168,9999.000000,0.085542,...,0.051348,0.131712,0.079870,0.072621,0.089225,0.091765,0.134046,0.114016,0.175972,0.099162
162,0.052702,0.058075,0.092768,0.031257,0.069676,0.034030,0.107526,0.106379,0.085542,9999.000000,...,0.042442,0.062744,0.097524,0.080594,0.064197,0.107016,0.065925,0.127734,0.102193,0.105615


,297,396,145,67,420,267,397,175,398,314,...,174,16,399,255,154,394,26,52,209,29
297,9999.000000,0.050637,0.187698,0.178725,0.138663,0.223895,0.137646,0.176154,0.088488,0.102930,...,0.102930,0.197372,0.169196,0.142655,0.139313,0.021674,0.178023,0.169731,0.041660,0.179175
396,0.050637,9999.000000,0.137100,0.132507,0.089527,0.267059,0.088460,0.146857,0.038090,0.058744,...,0.058744,0.147053,0.119546,0.101338,0.114769,0.066696,0.127390,0.124375,0.087322,0.128772
145,0.187698,0.137100,9999.000000,0.059925,0.061328,0.390837,0.061711,0.141631,0.100000,0.101662,...,0.101662,0.015521,0.029934,0.088676,0.143738,0.202854,0.010557,0.065635,0.222732,0.022770
67,0.178725,0.132507,0.059925,9999.000000,0.094929,0.357241,0.094567,0.083073,0.104505,0.123113,...,0.123113,0.052465,0.034801,0.042957,0.092765,0.197906,0.061770,0.010457,0.218827,0.078127
420,0.138663,0.089527,0.061328,0.094929,9999.000000,0.355334,0.001142,0.157934,0.052242,0.041552,...,0.041552,0.075907,0.062896,0.096530,0.144395,0.150352,0.050893,0.093681,0.168872,0.044289
267,0.223895,0.267059,0.390837,0.357241,0.355334,9999.000000,0.354219,0.302702,0.303121,0.324899,...,0.324899,0.395580,0.364710,0.314316,0.271478,0.222918,0.383178,0.346786,0.215249,0.389424
397,0.137646,0.088460,0.061711,0.094567,0.001142,0.354219,9999.000000,0.157138,0.051136,0.040873,...,0.040873,0.076204,0.062733,0.095730,0.143429,0.149409,0.051247,0.093202,0.167969,0.044966
175,0.176154,0.146857,0.141631,0.083073,0.157934,0.302702,0.157138,9999.000000,0.139107,0.169155,...,0.169155,0.135515,0.112936,0.061409,0.038277,0.197812,0.141248,0.076029,0.217339,0.156570
398,0.088488,0.038090,0.100000,0.104505,0.052242,0.303121,0.051136,0.139107,9999.000000,0.032747,...,0.032747,0.110925,0.085434,0.083098,0.114532,0.102859,0.090018,0.098129,0.122785,0.090715
314,0.102930,0.058744,0.101662,0.123113,0.041552,0.324899,0.040873,0.169155,0.032747,9999.000000,...,0.100000,0.115404,0.096561,0.110371,0.146662,0.111799,0.091106,0.118860,0.129237,0.085836


,395,240,371,372,181,259,210,72,81,31,...,365,304,128,426,48,275,256,39,355,397
395,9999.000000,0.120088,0.142655,0.146467,0.116648,0.041054,0.089083,0.044979,0.118672,0.131465,...,0.158863,0.024325,0.136939,0.148163,0.104350,0.128346,0.128346,16.573549,0.009327,0.095730
240,0.120088,9999.000000,0.034673,0.038753,0.109901,0.146038,0.094909,0.110866,0.156747,0.164631,...,0.046189,0.095997,0.175259,0.037795,0.116131,0.015765,0.015765,16.601393,0.111470,0.103085
371,0.142655,0.034673,9999.000000,0.004251,0.144459,0.161628,0.129319,0.141104,0.191218,0.199258,...,0.016473,0.118418,0.209853,0.005756,0.150766,0.019073,0.019073,16.633772,0.133481,0.137646
372,0.146467,0.038753,0.004251,9999.000000,0.148427,0.164888,0.133492,0.145311,0.195383,0.203375,...,0.013351,0.122262,0.213985,0.003986,0.154878,0.023245,0.023245,16.637113,0.137272,0.141787
181,0.116648,0.109901,0.144459,0.148427,9999.000000,0.157668,0.028426,0.073275,0.054504,0.058308,...,0.153645,0.107560,0.069344,0.146973,0.017614,0.125665,0.125665,16.495582,0.114896,0.020930
259,0.041054,0.146038,0.161628,0.164888,0.157668,9999.000000,0.129949,0.085747,0.157060,0.169884,...,0.178081,0.055831,0.174340,0.167374,0.145339,0.150796,0.150796,16.602517,0.043704,0.136761
210,0.089083,0.094909,0.129319,0.133492,0.028426,0.129949,9999.000000,0.047491,0.061910,0.070572,...,0.140740,0.079137,0.080854,0.132703,0.023274,0.110250,0.110250,16.518618,0.086818,0.009598
72,0.044979,0.110866,0.141104,0.145311,0.073275,0.085747,0.047491,9999.000000,0.076019,0.088559,...,0.155707,0.045052,0.095295,0.145833,0.059739,0.123428,0.123428,16.537095,0.045778,0.052484
81,0.118672,0.156747,0.191218,0.195383,0.054504,0.157060,0.061910,0.076019,9999.000000,0.012829,...,0.202384,0.120791,0.019989,0.194539,0.041601,0.172146,0.172146,16.462919,0.121218,0.053767
31,0.131465,0.164631,0.199258,0.203375,0.058308,0.169884,0.070572,0.088559,0.012829,9999.000000,...,0.209791,0.133161,0.011037,0.202334,0.048503,0.180204,0.180204,16.451374,0.133909,0.061729


,277,282,359,258,223,29,294,49,142,318,...,321,117,393,25,159,343,91,182,105,42
277,9999.000000,0.101269,0.063355,0.046190,0.026131,0.136900,0.117808,0.131862,0.161508,0.057721,...,0.105157,0.086097,0.060597,0.167300,0.097612,0.069982,0.118011,0.030910,0.131547,0.117032
282,0.101269,9999.000000,0.150427,0.060408,0.079837,0.056736,0.162512,0.057577,0.065380,0.116465,...,0.004433,0.091143,0.148632,0.072330,0.131708,0.165643,0.016742,0.112708,0.036605,0.067151
359,0.063355,0.150427,9999.000000,0.106270,0.088478,0.195214,0.168045,0.191171,0.215027,0.113596,...,0.154785,0.149080,0.002982,0.221544,0.154427,0.023427,0.166409,0.085130,0.185065,0.179080
258,0.046190,0.060408,0.106270,9999.000000,0.020870,0.090715,0.114532,0.085802,0.116200,0.059611,...,0.063612,0.056246,0.103804,0.121710,0.086205,0.115828,0.076628,0.052303,0.086580,0.073070
223,0.026131,0.079837,0.088478,0.020870,9999.000000,0.111148,0.109388,0.105912,0.137066,0.049825,...,0.083335,0.063959,0.085840,0.142577,0.084298,0.096113,0.096372,0.034296,0.107400,0.091004
29,0.136900,0.056736,0.195214,0.090715,0.111148,9999.000000,0.153799,0.008192,0.036134,0.127370,...,0.053090,0.084045,0.192969,0.037921,0.124250,0.206395,0.051160,0.136180,0.027161,0.035374
294,0.117808,0.162512,0.168045,0.114532,0.109388,0.153799,9999.000000,0.145621,0.189820,0.060622,...,0.163677,0.072754,0.165096,0.191544,0.030946,0.160003,0.174447,0.087338,0.169163,0.118455
49,0.131862,0.057577,0.191171,0.085802,0.105912,0.008192,0.145621,9999.000000,0.044322,0.120048,...,0.054340,0.076113,0.188855,0.046021,0.116127,0.201630,0.054336,0.129795,0.032429,0.027185
142,0.161508,0.065380,0.215027,0.116200,0.137066,0.036134,0.189820,0.044322,9999.000000,0.160293,...,0.060946,0.119262,0.213096,0.007425,0.160052,0.228948,0.051357,0.165698,0.030110,0.071484
318,0.057721,0.116465,0.113596,0.059611,0.049825,0.127370,0.060622,0.120048,0.160293,9999.000000,...,0.118807,0.048459,0.110617,0.164109,0.040921,0.110784,0.131221,0.028695,0.133833,0.096324


,234,109,210,24,208,238,39,272,371,394,...,123,216,396,199,307,314,192,120,101,315
234,9999.000000,0.137637,0.135207,0.175059,0.070615,0.130553,16.630368,0.112986,0.017660,0.006598,...,0.162025,0.181805,0.060203,0.097524,0.144925,0.105454,0.244535,0.196904,0.233940,0.193056
109,0.137637,9999.000000,0.063638,0.071740,0.173637,0.009197,16.564254,0.071892,0.124626,0.143831,...,0.071949,0.067274,0.078982,0.051104,0.079696,0.082955,0.253405,0.059308,0.114567,0.087420
210,0.135207,0.063638,9999.000000,0.041428,0.143221,0.067733,16.518618,0.024217,0.129319,0.141766,...,0.026889,0.129708,0.079682,0.043893,0.016933,0.036364,0.189969,0.096305,0.101120,0.150410
24,0.175059,0.071740,0.041428,9999.000000,0.184046,0.079672,16.493423,0.065621,0.167707,0.181653,...,0.016527,0.125720,0.117160,0.079175,0.040777,0.077348,0.202696,0.074077,0.059720,0.146772
208,0.070615,0.173637,0.143221,0.184046,9999.000000,0.169378,16.582813,0.119281,0.086189,0.071146,...,0.168038,0.231376,0.099595,0.123316,0.145075,0.106867,0.187669,0.228701,0.242879,0.246398
238,0.130553,0.009197,0.067733,0.079672,0.169378,9999.000000,16.572714,0.073018,0.117036,0.136646,...,0.078677,0.066743,0.072876,0.048653,0.084286,0.083193,0.256521,0.066979,0.123723,0.085975
39,16.630368,16.564254,16.518618,16.493423,16.582813,16.572714,9999.000000,16.532283,16.633772,16.635388,...,16.497719,16.598333,16.592771,16.562007,16.503088,16.534149,16.395993,16.533757,16.454981,16.614726
272,0.112986,0.071892,0.024217,0.065621,0.119281,0.073018,16.532283,9999.000000,0.108656,0.119473,...,0.050565,0.139088,0.061175,0.032597,0.032053,0.013026,0.185090,0.115158,0.125267,0.158865
371,0.017660,0.124626,0.129319,0.167707,0.086189,0.117036,16.633772,0.108656,9999.000000,0.021674,...,0.155738,0.165506,0.050637,0.088781,0.140667,0.102930,0.253184,0.183890,0.225605,0.176154
394,0.006598,0.143831,0.141766,0.181653,0.071146,0.136646,16.635388,0.119473,0.021674,9999.000000,...,0.168593,0.186974,0.066696,0.104099,0.151384,0.111799,0.248197,0.203122,0.240537,0.197812


,395,124,275,324,34,187,388,298,414,330,...,0,359,254,255,270,213,170,95,138,323
395,9999.000000,0.101856,0.128346,0.074121,0.046172,0.060541,0.039239,0.161187,0.029898,0.024325,...,0.102687,0.168944,0.163080,0.100000,0.009596,0.110648,0.047675,0.135801,0.127695,0.050131
124,0.101856,9999.000000,0.125277,0.117960,0.076385,0.151370,0.141050,0.150524,0.106839,0.094462,...,0.017868,0.157448,0.155516,0.101856,0.104060,0.060510,0.138248,0.076371,0.064783,0.100025
275,0.128346,0.125277,9999.000000,0.062071,0.150248,0.123064,0.147602,0.032874,0.102498,0.104026,...,0.141737,0.040682,0.035356,0.128346,0.120706,0.068025,0.118156,0.200268,0.188689,0.079190
324,0.074121,0.117960,0.062071,9999.000000,0.108543,0.061754,0.086362,0.093015,0.044810,0.051339,...,0.129317,0.100302,0.093590,0.074121,0.065068,0.084567,0.056087,0.181332,0.170654,0.025812
34,0.046172,0.076385,0.150248,0.108543,9999.000000,0.106711,0.079890,0.182328,0.070840,0.059237,...,0.069933,0.190156,0.185509,0.046172,0.054352,0.110346,0.093805,0.090905,0.083834,0.082755
187,0.060541,0.151370,0.123064,0.061754,0.106711,9999.000000,0.040067,0.152265,0.044531,0.057157,...,0.156589,0.159036,0.151882,0.060541,0.053005,0.137765,0.013476,0.195682,0.187047,0.058023
388,0.039239,0.141050,0.147602,0.086362,0.079890,0.040067,9999.000000,0.179272,0.047377,0.053092,...,0.141729,0.186630,0.179952,0.039239,0.038089,0.144248,0.034175,0.170756,0.163643,0.070341
298,0.161187,0.150524,0.032874,0.093015,0.182328,0.152265,0.179272,9999.000000,0.134962,0.136863,...,0.167799,0.007852,0.006032,0.161187,0.153458,0.090387,0.148656,0.226699,0.215080,0.111808
414,0.029898,0.106839,0.102498,0.044810,0.070840,0.044531,0.047377,0.134962,9999.000000,0.013238,...,0.112476,0.142570,0.136312,0.029898,0.020486,0.098445,0.031536,0.154969,0.145593,0.023504
330,0.024325,0.094462,0.104026,0.051339,0.059237,0.057157,0.053092,0.136863,0.013238,9999.000000,...,0.099494,0.144620,0.138768,0.024325,0.017362,0.091184,0.043868,0.141845,0.132401,0.026257


,179,41,112,33,191,158,283,166,3,297,...,253,57,403,346,295,53,46,25,186,310
179,9999.000000,0.140455,0.104932,0.030242,0.114366,0.132934,0.095094,0.008351,0.119022,0.131664,...,0.137393,0.044762,0.357404,0.156292,0.159328,0.112051,0.150894,0.087466,0.180738,0.156319
41,0.140455,9999.000000,0.035583,0.131625,0.075166,0.207958,0.082071,0.145779,0.076032,0.191368,...,0.212977,0.101321,0.368586,0.065973,0.088692,0.089870,0.017271,0.114320,0.088149,0.097266
112,0.104932,0.035583,9999.000000,0.097708,0.059887,0.180504,0.058329,0.110409,0.066385,0.165973,...,0.185609,0.067198,0.358901,0.073967,0.090877,0.075762,0.048055,0.090698,0.100183,0.095826
33,0.030242,0.131625,0.097708,9999.000000,0.122935,0.162445,0.105349,0.027234,0.095551,0.159824,...,0.167008,0.030511,0.383984,0.160213,0.167594,0.086294,0.139078,0.058332,0.186090,0.166632
191,0.114366,0.075166,0.059887,0.122935,9999.000000,0.136863,0.019658,0.122274,0.125571,0.118418,...,0.141687,0.097146,0.299072,0.045277,0.045080,0.133139,0.092433,0.139891,0.067055,0.043868
158,0.132934,0.207958,0.180504,0.162445,0.136863,9999.000000,0.125952,0.138732,0.230047,0.022390,...,0.005124,0.161865,0.232183,0.175148,0.159308,0.229394,0.224578,0.214740,0.185240,0.148656
283,0.095094,0.082071,0.058329,0.105349,0.019658,0.125952,9999.000000,0.103099,0.120147,0.109439,...,0.130951,0.081359,0.303526,0.064416,0.064254,0.125619,0.098877,0.127854,0.086713,0.061642
166,0.008351,0.145779,0.110409,0.027234,0.122274,0.138732,0.103099,9999.000000,0.120116,0.138372,...,0.143075,0.047488,0.364538,0.163794,0.167289,0.112154,0.155588,0.085531,0.188442,0.164438
3,0.119022,0.076032,0.066385,0.095551,0.125571,0.230047,0.120147,0.120116,9999.000000,0.219433,...,0.235129,0.075050,0.423526,0.136866,0.156190,0.015660,0.070938,0.049592,0.161693,0.161916
297,0.131664,0.191368,0.165973,0.159824,0.118418,0.022390,0.109439,0.138372,0.219433,9999.000000,...,0.025871,0.155081,0.226863,0.154755,0.137995,0.220180,0.208287,0.208681,0.163717,0.127131


,251,316,400,381,225,42,90,212,409,11,...,320,401,111,219,208,35,144,274,385,264
251,9999.000000,0.136033,0.111475,0.134698,0.154931,0.106330,0.160959,0.175743,0.029882,0.117455,...,0.059403,0.009576,0.160959,0.167057,0.209218,0.175709,0.079870,0.035257,0.133646,0.016949
316,0.136033,9999.000000,0.067251,0.032763,0.135453,0.060992,0.067872,0.150729,0.143047,0.068689,...,0.076879,0.145577,0.067872,0.137618,0.131208,0.099199,0.104156,0.101699,0.002836,0.119337
400,0.111475,0.067251,9999.000000,0.040500,0.072612,0.100747,0.131473,0.091052,0.131632,0.114203,...,0.063268,0.119505,0.131473,0.078365,0.100157,0.160979,0.124398,0.078737,0.064635,0.095745
381,0.134698,0.032763,0.040500,9999.000000,0.103501,0.085676,0.100635,0.118147,0.148242,0.096237,...,0.076876,0.143797,0.100635,0.105052,0.101658,0.131905,0.122370,0.099489,0.030953,0.117812
225,0.154931,0.135453,0.072612,0.103501,9999.000000,0.173026,0.202384,0.021038,0.181971,0.186685,...,0.126372,0.159947,0.202384,0.013631,0.076826,0.232684,0.191576,0.131715,0.133206,0.143619
42,0.106330,0.060992,0.100747,0.085676,0.173026,9999.000000,0.054672,0.191799,0.101135,0.014460,...,0.061995,0.115338,0.054672,0.179080,0.187223,0.072350,0.046795,0.080960,0.060623,0.092930
90,0.160959,0.067872,0.131473,0.100635,0.202384,0.054672,9999.000000,0.218353,0.154824,0.045598,...,0.112441,0.170003,0.100000,0.205238,0.195872,0.031688,0.094918,0.134028,0.069768,0.147196
212,0.175743,0.150729,0.091052,0.118147,0.021038,0.191799,0.218353,9999.000000,0.202948,0.205195,...,0.146983,0.180582,0.218353,0.013116,0.069959,0.249152,0.211983,0.152745,0.148669,0.164610
409,0.029882,0.143047,0.131632,0.148242,0.181971,0.101135,0.154824,0.202948,9999.000000,0.109398,...,0.071605,0.030957,0.154824,0.193587,0.231376,0.164476,0.063744,0.053072,0.141031,0.038379
11,0.117455,0.068689,0.114203,0.096237,0.186685,0.014460,0.045598,0.205195,0.109398,9999.000000,...,0.076108,0.126170,0.045598,0.192389,0.197856,0.059058,0.049819,0.094076,0.068849,0.104954


## Solve TSP 40 addresses with GAMS

In [ ]:

def solve_tsp(c_df, sub):
    m = Container()

    c_df.reset_index(inplace=True, drop=True)
    c_df.columns = range(c_df.shape[1])

    # Define nodes
    nodos = [str(i) for i in c_df.index]


    # Main node set
    ii = Set(m, name="ii", domain="*", records=nodos)

    # Aliases to allow i,j indexing over the same node set
    jj = Alias(m, "jj", alias_with=ii)
    i = Alias(m, "i", alias_with=ii)
    j = Alias(m, "j", alias_with=ii)

    records = c_df.stack().reset_index()
    records.columns = ['i', 'j', 'value']

    # Cost parameter c(i,j) (Distance traveled)
    c = gp.Parameter(m, "c", domain=[ii, jj], records=records)


    # Binary variable: x(i,j) = 1 if path goes from i to j
    x = Variable(m, name="x", type="binary", domain=[i, j])

    # Continuous auxiliary variable for subtour elimination (MTZ formulation)
    u = Variable(m, name="u", type = "positive", domain=ii)

    # Define equations.
    rowsum = Equation(m, name="rowsum", domain=ii)
    colsum = Equation(m, name="colsum", domain=jj)
    subtour = Equation(m, name="subtour", domain=[ii,jj])

    # Each node must leave exactly once
    rowsum[i] = Sum(j, x[i, j]) == 1
    # Each node must be entered exactly once
    colsum[j] = Sum(i, x[i, j]) == 1

    # Miller–Tucker–Zemlin (MTZ) subtour elimination constraints
    subtour = Equation(m, name="subtour", domain=[ii, jj])
    subtour[i, j].where[(Ord(i) > 1) & (Ord(j) > 1) & (Ord(i) != Ord(j))] = u[i] - u[j] + Card(i) * x[i, j] <= Card(i) - 1

    # Minimize total traveled distance
    objective = Sum([i, j], c[i, j] * x[i, j])

    assign = Model(
        m,
        name="tsp",
        equations=[rowsum, colsum, subtour],
        problem="mip",
        sense="min",
        objective=objective,
    )

    sol = assign.solve(options= Options(relative_optimality_gap=0.0001,time_limit=900, threads=6))

    # Retrieve all x(i,j) variable records
    camino = x.records
    mat = x.records

    # Filter arcs that are part of the optimal tour (x = 1)
    camino = camino[camino["level"] == 1]
    camino.reset_index(inplace=True, drop= True)


    sub_list = sub.tolist()

    # Convert node labels to integers for mapping
    mat["i"] = mat["i"].astype(int)
    mat["j"] = mat["j"].astype(int)
    camino["i"] = camino["i"].astype(int)
    camino["j"] = camino["j"].astype(int)


    # Map internal indices back to original unique address identifiers indexes.
    mat["i"] = mat["i"].map(lambda x: sub_list[x])
    mat["j"] = mat["j"].map(lambda x: sub_list[x])
    camino["i"] = camino["i"].map(lambda x: sub_list[x])
    camino["j"] = camino["j"].map(lambda x: sub_list[x])

    # Convert identifiers back to strings
    mat["i"] = mat["i"].astype(str)
    mat["j"] = mat["j"].astype(str)
    camino["i"] = camino["i"].astype(str)
    camino["j"] = camino["j"].astype(str)

    #print(camino["i"].value_counts())
    #print(camino["j"].value_counts())
    #print(camino.shape[0])

    # Print distance traveled by traveler.
    print(f"Costo: {sol["Objective"]}")

    # Reconstruct path followed to a list.
    secuencia = reconstruct_tsp_path_safe(camino)

    return mat, camino, secuencia, sol["Objective"], sol["Solver Time"]




In [ ]:
resultados_40 = {}

# Solve each 40-node instance, and store the transition matrices, tour, objective value, and solution time inside a dictionary.
for idx, matriz in enumerate(matrices_40):
    print(f"✅ Resolviendo instancia {idx+1}")
    print(matriz.columns)

    mat, camino, sec, obj, t= solve_tsp(matriz, matriz.columns)

    count(sec)

    resultados_40[idx] = {
        "mat": mat,
        "camino": camino,
        "secuencia": sec,
        "objetivo": obj,
        "time": t
    }

    print(f"✅ Finalizado con instancia {idx+1}, {camino.shape[0]}, {sec},  (Objetivo: {obj})")



✅ Resolviendo instancia 1
RangeIndex(start=0, stop=40, step=1)


/tmp/ipython-input-306832568.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  camino["i"] = camino["i"].astype(int)
/tmp/ipython-input-306832568.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  camino["j"] = camino["j"].astype(int)
/tmp/ipython-input-306832568.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user

Costo: 0    132.922945
Name: Objective, dtype: float64
Valor 0 se repite 2 veces
Cantidad de valores únicos: 40
✅ Finalizado con instancia 1, 40, ['0', '7', '36', '19', '32', '33', '34', '25', '28', '16', '31', '13', '10', '4', '1', '22', '18', '12', '35', '3', '5', '8', '29', '6', '26', '2', '23', '20', '30', '17', '27', '21', '14', '39', '37', '11', '38', '24', '15', '9', '0'],  (Objetivo: 0    132.922945
Name: Objective, dtype: float64)
✅ Resolviendo instancia 2
RangeIndex(start=0, stop=40, step=1)


/tmp/ipython-input-306832568.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  camino["i"] = camino["i"].astype(int)
/tmp/ipython-input-306832568.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  camino["j"] = camino["j"].astype(int)
/tmp/ipython-input-306832568.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user

Costo: 0    1.324939
Name: Objective, dtype: float64
Valor 0 se repite 2 veces
Cantidad de valores únicos: 40
✅ Finalizado con instancia 2, 40, ['0', '17', '32', '23', '8', '7', '27', '37', '31', '39', '15', '2', '34', '6', '12', '21', '14', '38', '22', '9', '18', '26', '33', '11', '29', '5', '16', '20', '3', '19', '35', '30', '25', '13', '36', '24', '28', '1', '4', '10', '0'],  (Objetivo: 0    1.324939
Name: Objective, dtype: float64)
✅ Resolviendo instancia 3
RangeIndex(start=0, stop=40, step=1)


...

/tmp/ipython-input-306832568.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  camino["i"] = camino["i"].astype(int)
/tmp/ipython-input-306832568.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  camino["j"] = camino["j"].astype(int)
/tmp/ipython-input-306832568.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user

Costo: 0    1.028014
Name: Objective, dtype: float64
Valor 0 se repite 2 veces
Cantidad de valores únicos: 40
✅ Finalizado con instancia 8, 40, ['0', '23', '11', '18', '4', '20', '26', '16', '13', '29', '37', '38', '12', '30', '1', '24', '19', '35', '10', '27', '22', '14', '2', '7', '32', '31', '21', '15', '3', '39', '9', '8', '36', '5', '28', '6', '17', '25', '33', '34', '0'],  (Objetivo: 0    1.028014
Name: Objective, dtype: float64)
✅ Resolviendo instancia 9
RangeIndex(start=0, stop=40, step=1)


[MODEL - WARNING] The solve was interrupted! Solve status: UserInterrupt. For further information, see https://gamspy.readthedocs.io/en/latest/reference/gamspy._model.html#gamspy.SolveStatus.
/tmp/ipython-input-306832568.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  camino["i"] = camino["i"].astype(int)
/tmp/ipython-input-306832568.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  camino["j"] = camino["j"].astype(int)
/tmp/ipython-input-306832568.py:78: SettingWithCopyWarning: 
A value is trying t

Costo: 0    1.377894
Name: Objective, dtype: float64
Valor 0 se repite 2 veces
Cantidad de valores únicos: 40
✅ Finalizado con instancia 9, 40, ['0', '13', '7', '27', '3', '16', '20', '31', '21', '11', '37', '35', '8', '36', '1', '2', '6', '18', '29', '4', '23', '33', '15', '34', '39', '26', '38', '19', '28', '32', '30', '12', '24', '5', '22', '9', '17', '10', '25', '14', '0'],  (Objetivo: 0    1.377894
Name: Objective, dtype: float64)
✅ Resolviendo instancia 10
Index([251, 316, 400, 381, 225,  42,  90, 212, 409,  11, 337, 295, 140, 150,
       123, 227, 253, 197, 396, 141, 217,  46,  50, 426, 168, 315, 266,  69,
       280,  79, 320, 401, 111, 219, 208,  35, 144, 274, 385, 264],
      dtype='int64')
Costo: 0    0.982061
Name: Objective, dtype: float64
Valor 251 se repite 2 veces
Cantidad de valores únicos: 40
✅ Finalizado con instancia 10, 40, ['251', '337', '150', '168', '197', '401', '315', '409', '217', '50', '141', '144', '46', '69', '140', '35', '90', '79', '111', '11', '42', '12

/tmp/ipython-input-306832568.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  camino["i"] = camino["i"].astype(int)
/tmp/ipython-input-306832568.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  camino["j"] = camino["j"].astype(int)
/tmp/ipython-input-306832568.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user

In [ ]:
import pandas as pd

# Convert dictionary to DataFrame.
df_resultados = pd.DataFrame([
    {"Instancia": i + 1,
     "Secuencia": resultados_40[i]["secuencia"],
     "Objetivo": resultados_40[i]["objetivo"],
     "Time": resultados_40[i]["time"],}
    for i in range(10)
])

display(df_resultados)


,Instancia,Secuencia,Objetivo,Time
0,1,"[0, 7, 36, 19, 32, 33, 34, 25, 28, 16, 31, 13,...","0 132.922945 Name: Objective, dtype: float64","0 45.915 Name: Solver Time, dtype: float64"
1,2,"[0, 17, 32, 23, 8, 7, 27, 37, 31, 39, 15, 2, 3...","0 1.324939 Name: Objective, dtype: float64","0 2.901 Name: Solver Time, dtype: float64"
2,3,"[0, 30, 22, 8, 21, 11, 14, 10, 18, 24, 34, 23,...","0 0.929544 Name: Objective, dtype: float64","0 76.063 Name: Solver Time, dtype: float64"
3,4,"[0, 16, 26, 1, 8, 19, 9, 25, 30, 12, 21, 6, 4,...","0 133.039967 Name: Objective, dtype: float64","0 69.246 Name: Solver Time, dtype: float64"
4,5,"[0, 11, 38, 31, 29, 7, 10, 12, 28, 25, 24, 18,...","0 33.906638 Name: Objective, dtype: float64","0 6.196 Name: Solver Time, dtype: float64"
5,6,"[0, 18, 22, 14, 32, 2, 35, 37, 9, 15, 17, 13, ...","0 0.971227 Name: Objective, dtype: float64","0 14.649 Name: Solver Time, dtype: float64"
6,7,"[0, 16, 8, 18, 32, 12, 24, 14, 23, 33, 7, 35, ...","0 33.992213 Name: Objective, dtype: float64","0 67.099 Name: Solver Time, dtype: float64"
7,8,"[0, 23, 11, 18, 4, 20, 26, 16, 13, 29, 37, 38,...","0 1.028014 Name: Objective, dtype: float64","0 38.674 Name: Solver Time, dtype: float64"
8,9,"[0, 13, 7, 27, 3, 16, 20, 31, 21, 11, 37, 35, ...","0 1.377894 Name: Objective, dtype: float64","0 133.158 Name: Solver Time, dtype: float64"
9,10,"[251, 337, 150, 168, 197, 401, 315, 409, 217, ...","0 0.982061 Name: Objective, dtype: float64","0 80.803 Name: Solver Time, dtype: float64"


In [ ]:
# Export results to an excel file.
df_resultados.to_excel("resultados_40.xlsx")

# Deine solver Function using NEOS optimization servers.

Due to the problem complexity and the limitations of the GAMSpy free license, instances with 100, 150, 200, and 250 nodes (addresses) were solved using a cloud-based solver, which removed the memory restrictions imposed by the free license. The cloud service used for this purpose was the NEOS optimization server.

In [ ]:
def solve_tsp_neos(c_df, sub, n = False):
    m = Container()

    c_df.reset_index(inplace=True, drop=True)
    c_df.columns = range(c_df.shape[1])

    # Define nodes
    nodos = [str(i) for i in c_df.index]


    # Main node set
    ii = Set(m, name="ii", domain="*", records=nodos)

    # Aliases to allow i,j indexing over the same node set
    jj = Alias(m, "jj", alias_with=ii)
    i = Alias(m, "i", alias_with=ii)
    j = Alias(m, "j", alias_with=ii)

    records = c_df.stack().reset_index()
    records.columns = ['i', 'j', 'value']

    # Cost parameter c(i,j) (Distance traveled)
    c = gp.Parameter(m, "c", domain=[ii, jj], records=records)


    # Binary variable: x(i,j) = 1 if path goes from i to j
    x = Variable(m, name="x", type="binary", domain=[i, j])

    # Continuous auxiliary variable for subtour elimination (MTZ formulation
    u = Variable(m, name="u", type = "positive", domain=ii)

    # Define equations.
    rowsum = Equation(m, name="rowsum", domain=ii)
    colsum = Equation(m, name="colsum", domain=jj)
    subtour = Equation(m, name="subtour", domain=[ii,jj])

    # Each node must leave exactly once
    rowsum[i] = Sum(j, x[i, j]) == 1
    # Each node must be entered exactly once
    colsum[j] = Sum(i, x[i, j]) == 1

    # Miller–Tucker–Zemlin (MTZ) subtour elimination constraints
    subtour = Equation(m, name="subtour", domain=[ii, jj])
    subtour[i, j].where[(Ord(i) > 1) & (Ord(j) > 1) & (Ord(i) != Ord(j))] = u[i] - u[j] + Card(i) * x[i, j] <= Card(i) - 1

    # Minimize total traveled distance
    objective = Sum([i, j], c[i, j] * x[i, j])

    assign = Model(
        m,
        name="tsp",
        equations=[rowsum, colsum, subtour],
        problem="mip",
        sense="min",
        objective=objective,
    )

    # Log into NEOS
    client= gp.NeosClient(email="A01198941@tec.mx")
    sol = assign.solve(backend="neos", client=client, options= Options(relative_optimality_gap=0.05, time_limit=600))

    # Retrieve all x(i,j) variable records
    camino = x.records
    mat = x.records

    # Filter arcs that are part of the optimal tour (x = 1)
    camino = camino[camino["level"] == 1]
    camino.reset_index(inplace=True, drop= True)

    sub_list = sub.tolist()

    # Convert node labels to integers for mapping
    mat["i"] = mat["i"].astype(int)
    mat["j"] = mat["j"].astype(int)
    camino["i"] = camino["i"].astype(int)
    camino["j"] = camino["j"].astype(int)

    # Map internal indices back to original unique address identifiers indexes.
    mat["i"] = mat["i"].map(lambda x: sub_list[x])
    mat["j"] = mat["j"].map(lambda x: sub_list[x])
    camino["i"] = camino["i"].map(lambda x: sub_list[x])
    camino["j"] = camino["j"].map(lambda x: sub_list[x])

    # Convert identifiers back to strings
    mat["i"] = mat["i"].astype(str)
    mat["j"] = mat["j"].astype(str)
    camino["i"] = camino["i"].astype(str)
    camino["j"] = camino["j"].astype(str)

    #print(camino["i"].value_counts())
    #print(camino["j"].value_counts())
    #print(camino.shape[0])

    # Print distance traveled by traveler.
    print(f"Costo: {sol["Objective"]}")

    # Reconstruct path followed to a list.
    secuencia = reconstruct_tsp_path_safe(camino)

    return mat, camino, secuencia, sol["Objective"], sol["Solver Time"]




# Solving instances of 100 addresses

## Generate distance sub matrices (matrices_100.xlsx).

In [ ]:
# Generate 10 instances of sub matrices consisting of 100 randomly chosen addresses. (matrices_100.xlsx)
matrices_100, subconjuntos = generar_submatrices_distancias(df_matriz_dist, n_subconjuntos=10, tamaño_sub=100)

✅ Subconjunto 1: 100 ubicaciones seleccionadas
✅ Subconjunto 2: 100 ubicaciones seleccionadas
✅ Subconjunto 3: 100 ubicaciones seleccionadas
✅ Subconjunto 4: 100 ubicaciones seleccionadas
✅ Subconjunto 5: 100 ubicaciones seleccionadas
✅ Subconjunto 6: 100 ubicaciones seleccionadas
✅ Subconjunto 7: 100 ubicaciones seleccionadas
✅ Subconjunto 8: 100 ubicaciones seleccionadas
✅ Subconjunto 9: 100 ubicaciones seleccionadas
✅ Subconjunto 10: 100 ubicaciones seleccionadas


In [ ]:
matrices_100[0]

,164,225,312,200,173,161,283,191,28,15,...,139,400,371,282,216,348,311,398,116,249
164,9999.000000,0.134587,0.054352,0.052702,0.025420,0.099115,0.019658,0.100000,0.054556,0.120791,...,0.109878,0.075589,0.118418,0.106450,0.057396,0.175745,0.077484,0.063364,0.055323,0.061977
225,0.134587,9999.000000,0.172501,0.115604,0.112831,0.147607,0.124996,0.134587,0.177200,0.202384,...,0.177766,0.072612,0.016473,0.149376,0.181971,0.076826,0.119791,0.101090,0.188422,0.099082
312,0.054352,0.172501,9999.000000,0.106976,0.078087,0.150625,0.073983,0.054352,0.082798,0.160570,...,0.155842,0.125819,0.156081,0.158156,0.012842,0.224754,0.131484,0.117441,0.062418,0.115866
200,0.052702,0.115604,0.106976,9999.000000,0.033122,0.052027,0.033048,0.052702,0.066520,0.092978,...,0.073066,0.043186,0.102193,0.058447,0.108860,0.135039,0.025204,0.016296,0.086423,0.017264
173,0.025420,0.112831,0.078087,0.033122,9999.000000,0.084427,0.012541,0.025420,0.066365,0.117481,...,0.101568,0.050262,0.097144,0.091171,0.082396,0.150393,0.058125,0.039858,0.075691,0.038016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
348,0.175745,0.076826,0.224754,0.135039,0.150393,0.139345,0.159562,0.175745,0.201240,0.195872,...,0.169517,0.100157,0.086189,0.137083,0.231376,9999.000000,0.123889,0.118994,0.220928,0.118853
311,0.077484,0.119791,0.131484,0.025204,0.058125,0.031046,0.057927,0.077484,0.080252,0.082601,...,0.058570,0.049678,0.108808,0.035996,0.132602,0.123889,9999.000000,0.024435,0.104233,0.027396
398,0.063364,0.101090,0.117441,0.016296,0.039858,0.055389,0.044261,0.063364,0.082803,0.103540,...,0.081219,0.028552,0.088488,0.060408,0.120647,0.118994,0.024435,9999.000000,0.102074,0.002963
116,0.055323,0.188422,0.062418,0.086423,0.075691,0.111401,0.063932,0.055323,0.028080,0.104237,...,0.106146,0.123201,0.172556,0.118769,0.053423,0.220928,0.104233,0.102074,9999.000000,0.102084


In [ ]:
# Save matrices to an excel file for later use
archivo_salida = "matrices_100.xlsx"

with pd.ExcelWriter(archivo_salida, engine="xlsxwriter") as writer:
    for idx, matriz in enumerate(matrices_100):
        hoja_nombre = f"Instancia_{idx+1}"
        matriz.to_excel(writer, sheet_name=hoja_nombre, index=True)


## Read pre-generated submatrices

In [11]:
archivo = "/content/tsp-bio-inspired-algorithms/Data_for_algorithms/matrices_100.xlsx"

# Read file.
hojas_dict = pd.read_excel(archivo, sheet_name=None, index_col=0)

# Convert lists into dataframes.
matrices_100 = [hojas_dict[f"Instancia_{i+1}"] for i in range(10)]

# Validation.
print(len(matrices_100))  # print number of instances 10.
print(matrices_100[0].head())  # print first instance.
print(matrices_100[0].shape) # print first instance shape.

10
             164          225          312          200          173  \
164  9999.000000     0.134587     0.054352     0.052702     0.025420   
225     0.134587  9999.000000     0.172501     0.115604     0.112831   
312     0.054352     0.172501  9999.000000     0.106976     0.078087   
200     0.052702     0.115604     0.106976  9999.000000     0.033122   
173     0.025420     0.112831     0.078087     0.033122  9999.000000   

          161       283       191       28        15   ...       139  \
164  0.099115  0.019658  0.100000  0.054556  0.120791  ...  0.109878   
225  0.147607  0.124996  0.134587  0.177200  0.202384  ...  0.177766   
312  0.150625  0.073983  0.054352  0.082798  0.160570  ...  0.155842   
200  0.052027  0.033048  0.052702  0.066520  0.092978  ...  0.073066   
173  0.084427  0.012541  0.025420  0.066365  0.117481  ...  0.101568   

          400       371       282       216       348       311       398  \
164  0.075589  0.118418  0.106450  0.057396  0.175745 

## Solve TSP 100 addresses with NEOS

In [ ]:
resultados_100 = {}

# Solve each 100-node instance, and store the transition matrices, tour, objective value, and solution time inside a dictionary.
for idx, matriz in enumerate(matrices_100):
    print(f"✅ Resolviendo instancia {idx+1}")
    print(matriz.columns)

    mat, camino, sec, obj, t = solve_tsp_neos(matriz, matriz.columns, n = True)


    count(sec)

    resultados_100[idx] = {
        "mat": mat,
        "camino": camino,
        "secuencia": sec,
        "objetivo": obj,
        "time": t
    }

    print(f"✅ Finalizado con instancia {idx+1}, {camino.shape[0]}, {sec},  (Objetivo: {obj})")



✅ Resolviendo instancia 1
Index([164, 225, 312, 200, 173, 161, 283, 191,  28,  15, 141, 125, 234,  56,
       335,   5, 299, 174, 111, 379, 275,  52,   0,  38, 388,  92, 259, 258,
       202,  61,  43, 267, 169, 110,  47, 268,  41,  97, 214,  18,  14,  55,
       415,  35, 301,   6, 158,  75, 387, 291, 418,  27, 177,  22, 109, 428,
       339,  96, 121,  13, 108, 396, 133, 344, 228, 403, 182,  58, 181, 170,
       178,  21, 107, 157, 285, 215, 224, 199, 172, 143, 318, 404, 100, 411,
       328,  86,  89, 247, 332, 146, 139, 400, 371, 282, 216, 348, 311, 398,
       116, 249],
      dtype='int64')


[NEOS - INFO] Job Number: 17951477, Job Password: UXBsSLjo
INFO:NEOS:Job Number: 17951477, Job Password: UXBsSLjo


NeosClientException: Couldn't get output file from NEOS Server because: Output file does not exist

# Solving instances of 150 addresses

## Generate distance submatrices (matrices_150.xlsx)

In [ ]:
# Generate 10 instances of sub matrices consisting of 150 randomly chosen addresses. (matrices_100.xlsx)
matrices_150, subconjuntos = generar_submatrices_distancias(df_matriz_dist, n_subconjuntos=10, tamaño_sub=150)

✅ Subconjunto 1: 150 ubicaciones seleccionadas
✅ Subconjunto 2: 150 ubicaciones seleccionadas
✅ Subconjunto 3: 150 ubicaciones seleccionadas
✅ Subconjunto 4: 150 ubicaciones seleccionadas
✅ Subconjunto 5: 150 ubicaciones seleccionadas
✅ Subconjunto 6: 150 ubicaciones seleccionadas
✅ Subconjunto 7: 150 ubicaciones seleccionadas
✅ Subconjunto 8: 150 ubicaciones seleccionadas
✅ Subconjunto 9: 150 ubicaciones seleccionadas
✅ Subconjunto 10: 150 ubicaciones seleccionadas


In [ ]:
archivo_salida = "matrices_150.xlsx"

# Save matrices in excel file for later use
with pd.ExcelWriter(archivo_salida, engine="xlsxwriter") as writer:
    for idx, matriz in enumerate(matrices_150):
        hoja_nombre = f"Instancia_{idx+1}"
        matriz.to_excel(writer, sheet_name=hoja_nombre, index=True)


In [ ]:
matrices_150[0]

,413,37,40,332,4,360,189,206,317,107,...,184,270,382,390,381,305,60,376,211,57
413,9999.000000,0.174416,0.234961,0.262736,0.255511,0.036963,0.154548,0.190521,0.164766,0.214139,...,0.107770,0.171175,0.153624,0.105410,0.123637,0.079819,0.214141,0.139579,0.205749,0.186881
37,0.174416,9999.000000,0.080131,0.228316,0.098318,0.138504,0.035411,0.081512,0.059024,0.046395,...,0.088410,0.055036,0.047223,0.069051,0.072280,0.103877,0.064834,0.056989,0.110254,0.050095
40,0.234961,0.080131,9999.000000,0.188269,0.020574,0.202748,0.115528,0.152123,0.072167,0.088133,...,0.166329,0.131128,0.081337,0.134988,0.112424,0.176571,0.020862,0.095537,0.178805,0.048122
332,0.262736,0.228316,0.188269,9999.000000,0.193339,0.253184,0.253019,0.308658,0.170354,0.262986,...,0.272234,0.280030,0.185988,0.225103,0.183416,0.260817,0.183041,0.185090,0.337408,0.180791
4,0.255511,0.098318,0.020574,0.193339,9999.000000,0.223312,0.133464,0.165823,0.092258,0.099270,...,0.185639,0.146907,0.101890,0.155393,0.132797,0.196710,0.041372,0.116036,0.191390,0.068640
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305,0.079819,0.103877,0.176571,0.260817,0.196710,0.043656,0.077554,0.111099,0.116571,0.137968,...,0.028520,0.091517,0.100454,0.043381,0.082409,9999.000000,0.156968,0.090522,0.128496,0.131467
60,0.214141,0.064834,0.020862,0.183041,0.041372,0.182125,0.099942,0.141985,0.051571,0.082992,...,0.148180,0.118516,0.060527,0.114880,0.091583,0.156968,9999.000000,0.074679,0.169744,0.027270
376,0.139579,0.056989,0.095537,0.185090,0.116036,0.108656,0.069855,0.127355,0.026538,0.103024,...,0.091759,0.097979,0.014731,0.047396,0.017902,0.090522,0.074679,9999.000000,0.155403,0.047418
211,0.205749,0.110254,0.178805,0.337408,0.191390,0.171541,0.085549,0.028808,0.167053,0.092283,...,0.100125,0.057614,0.151822,0.131363,0.163403,0.128496,0.169744,0.155403,9999.000000,0.160148


## Read pre-generated submatrices

In [15]:

archivo = "/content/tsp-bio-inspired-algorithms/Data_for_algorithms/matrices_150.xlsx"

# Read Excel file with matrices.
hojas_dict = pd.read_excel(archivo, sheet_name=None, index_col=0)

matrices_150 = [hojas_dict[f"Instancia_{i+1}"] for i in range(10)]

# Validate
print(len(matrices_150))
print(matrices_150[0].head())
print(matrices_150[0].shape)

10
             413          37           40           332          4    \
413  9999.000000     0.174416     0.234961     0.262736     0.255511   
37      0.174416  9999.000000     0.080131     0.228316     0.098318   
40      0.234961     0.080131  9999.000000     0.188269     0.020574   
332     0.262736     0.228316     0.188269  9999.000000     0.193339   
4       0.255511     0.098318     0.020574     0.193339  9999.000000   

          360       189       206       317       107  ...       184  \
413  0.036963  0.154548  0.190521  0.164766  0.214139  ...  0.107770   
37   0.138504  0.035411  0.081512  0.059024  0.046395  ...  0.088410   
40   0.202748  0.115528  0.152123  0.072167  0.088133  ...  0.166329   
332  0.253184  0.253019  0.308658  0.170354  0.262986  ...  0.272234   
4    0.223312  0.133464  0.165823  0.092258  0.099270  ...  0.185639   

          270       382       390       381       305       60        376  \
413  0.171175  0.153624  0.105410  0.123637  0.079819 

## Solve TSP 150 addresses with NEOS

In [ ]:
resultados_150 = {}

# Solve each 150-node instance, and store the transition matrices, tour, objective value, and solution time inside a dictionary.
for idx, matriz in enumerate(matrices_150):
    print(f"✅ Resolviendo instancia {idx+1}")
    print(matriz.columns)

    mat, camino, sec, obj, t = solve_tsp_neos(matriz, matriz.columns, n = True)


    count(sec)

    resultados_150[idx] = {
        "mat": mat,
        "camino": camino,
        "secuencia": sec,
        "objetivo": obj,
        "time": t
    }

    print(f"✅ Finalizado con instancia {idx+1}, {camino.shape[0]}, {sec},  (Objetivo: {obj})")



# Solving instances of 200 addresses

## Generate distance submatrices (matrices_200.xlsx).

In [ ]:
# Generate 10 instances of sub matrices consisting of 150 randomly chosen addresses. (matrices_100.xlsx)
matrices_200, subconjuntos = generar_submatrices_distancias(df_matriz_dist, n_subconjuntos=10, tamaño_sub=200)

✅ Subconjunto 1: 200 ubicaciones seleccionadas
✅ Subconjunto 2: 200 ubicaciones seleccionadas
✅ Subconjunto 3: 200 ubicaciones seleccionadas
✅ Subconjunto 4: 200 ubicaciones seleccionadas
✅ Subconjunto 5: 200 ubicaciones seleccionadas
✅ Subconjunto 6: 200 ubicaciones seleccionadas
✅ Subconjunto 7: 200 ubicaciones seleccionadas
✅ Subconjunto 8: 200 ubicaciones seleccionadas
✅ Subconjunto 9: 200 ubicaciones seleccionadas
✅ Subconjunto 10: 200 ubicaciones seleccionadas


In [ ]:
archivo_salida = "matrices_200.xlsx"

# Save matrices in Excel file for later use.
with pd.ExcelWriter(archivo_salida, engine="xlsxwriter") as writer:
    for idx, matriz in enumerate(matrices_200):
        hoja_nombre = f"Instancia_{idx+1}"
        matriz.to_excel(writer, sheet_name=hoja_nombre, index=True)


In [ ]:
matrices_200[0]

,127,408,335,287,27,190,119,417,320,99,...,424,24,372,354,33,77,138,135,71,407
127,9999.000000,0.108063,0.161569,0.114270,0.055300,0.130711,0.111372,0.161093,0.122911,0.051919,...,0.099551,0.041212,0.206993,0.160260,0.057550,0.081072,0.006689,0.032954,0.044456,0.416465
408,0.108063,9999.000000,0.066935,0.045277,0.055073,0.049593,0.069167,0.057531,0.035141,0.056247,...,0.070922,0.069451,0.104050,0.085302,0.081694,0.040811,0.111998,0.116104,0.102653,0.308430
335,0.161569,0.066935,9999.000000,0.105861,0.116482,0.032395,0.067244,0.075888,0.038863,0.112193,...,0.136324,0.120371,0.104666,0.034223,0.147375,0.107572,0.163662,0.158194,0.139821,0.265420
287,0.114270,0.045277,0.105861,9999.000000,0.061106,0.094349,0.112835,0.056767,0.079919,0.071265,...,0.033796,0.086537,0.097225,0.129206,0.066529,0.034034,0.119989,0.133423,0.125723,0.315937
27,0.055300,0.055073,0.116482,0.061106,9999.000000,0.089637,0.085371,0.105845,0.078223,0.013785,...,0.056952,0.027012,0.151698,0.124303,0.035240,0.027073,0.060316,0.072387,0.066467,0.362752
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,0.081072,0.040811,0.107572,0.034034,0.027073,0.086452,0.093572,0.082060,0.072777,0.037840,...,0.037411,0.052914,0.126761,0.122667,0.041722,9999.000000,0.086536,0.099412,0.092391,0.341368
138,0.006689,0.111998,0.163662,0.119989,0.060316,0.132401,0.111329,0.165835,0.125219,0.055753,...,0.106006,0.043786,0.211933,0.160970,0.064190,0.086536,9999.000000,0.027287,0.040837,0.420195
135,0.032954,0.116104,0.158194,0.133423,0.072387,0.125904,0.098573,0.172828,0.121536,0.063013,...,0.125192,0.047876,0.219503,0.149948,0.086369,0.099412,0.027287,9999.000000,0.020009,0.420433
71,0.044456,0.102653,0.139821,0.125723,0.066467,0.107433,0.078781,0.160096,0.104139,0.054568,...,0.122572,0.039568,0.206695,0.130254,0.088195,0.092391,0.040837,0.020009,9999.000000,0.403406


## Read pre-generated submatrices

In [16]:
archivo = "/content/tsp-bio-inspired-algorithms/Data_for_algorithms/matrices_200.xlsx"

hojas_dict = pd.read_excel(archivo, sheet_name=None, index_col=0)

# Read Excel file with matrices
matrices_200 = [hojas_dict[f"Instancia_{i+1}"] for i in range(10)]

# Validate
print(len(matrices_200))
print(matrices_200[0].head())
print(matrices_200[0].shape)



10
             127          408          335          287          27   \
127  9999.000000     0.108063     0.161569     0.114270     0.055300   
408     0.108063  9999.000000     0.066935     0.045277     0.055073   
335     0.161569     0.066935  9999.000000     0.105861     0.116482   
287     0.114270     0.045277     0.105861  9999.000000     0.061106   
27      0.055300     0.055073     0.116482     0.061106  9999.000000   

          190       119       417       320       99   ...       424  \
127  0.130711  0.111372  0.161093  0.122911  0.051919  ...  0.099551   
408  0.049593  0.069167  0.057531  0.035141  0.056247  ...  0.070922   
335  0.032395  0.067244  0.075888  0.038863  0.112193  ...  0.136324   
287  0.094349  0.112835  0.056767  0.079919  0.071265  ...  0.033796   
27   0.089637  0.085371  0.105845  0.078223  0.013785  ...  0.056952   

          24        372       354       33        77        138       135  \
127  0.041212  0.206993  0.160260  0.057550  0.081072 

## Solve TSP 200 addresses with NEOS

In [ ]:
# Solve each 200-node instance, and store the transition matrices, tour, objective value, and solution time inside a dictionary.
resultados_200 = {}
print(f"✅ Resolviendo instancia {idx+1}")
print(matrices_200[2].columns)

mat, camino, sec, obj, t = solve_tsp_neos(matrices_200[2], matrices_200[2].columns, n = True)


count(sec)

resultados_200[idx] = {
    "mat": mat,
    "camino": camino,
    "secuencia": sec,
    "objetivo": obj,
    "time": t
}

print(f"✅ Finalizado con instancia {idx+1}, {camino.shape[0]}, {sec},  (Objetivo: {obj})")



✅ Resolviendo instancia 9
Index([131, 204, 230, 335, 250, 165, 115, 383, 225,  93,
       ...
       276,   8, 302, 366, 382, 167, 267, 256, 129,   7],
      dtype='int64', length=200)


[NEOS - INFO] Job Number: 17951780, Job Password: CxqfFMHp
INFO:NEOS:Job Number: 17951780, Job Password: CxqfFMHp


In [ ]:
print(f"✅ Resolviendo instancia {idx+1}")
print(matrices_200[2].columns)

mat, camino, sec, obj, t = solve_tsp_neos(matrices_200[3], matrices_200[3].columns, n = True)


count(sec)

resultados_200[idx] = {
    "mat": mat,
    "camino": camino,
    "secuencia": sec,
    "objetivo": obj,
    "time": t
}

print(f"✅ Finalizado con instancia {idx+1}, {camino.shape[0]}, {sec},  (Objetivo: {obj})")

